# LLaMA Factory free Llama fine-tuning

Run this notebook in Colab or Kaggle with GPU. It converts the bundled OpenAI-style JSONL into LLaMA Factory sharegpt format, runs QLoRA SFT, then uploads the adapter to Hugging Face.


In [ ]:
!nvidia-smi


In [ ]:
import os
from pathlib import Path

if not Path('/content/LLaMA-Factory').exists():
    !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git /content/LLaMA-Factory

%cd /content/LLaMA-Factory
!pip install -q -e .
!pip install -q -r requirements/metrics.txt
!pip install -q bitsandbytes


In [ ]:
import os
import getpass
from pathlib import Path

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Paste Hugging Face write token: ')

BASE_MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
OUTPUT_REPO = 'Phuoc20050911/copypro-brand-voice-llama-1b-lora-llamafactory'
SOURCE_JSONL = Path('/content/02_train_huggingface_chat_utf8.jsonl')

assert SOURCE_JSONL.exists(), f'Missing {SOURCE_JSONL}. Upload it to Colab first.'
print('Base model:', BASE_MODEL)
print('Output repo:', OUTPUT_REPO)


In [ ]:
import json
import shutil
from pathlib import Path

factory = Path('/content/LLaMA-Factory')
data_dir = factory / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
sharegpt_path = data_dir / 'copypro_vi_sharegpt.json'

records = []
for line in SOURCE_JSONL.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line:
        continue
    item = json.loads(line)
    system = ''
    user = ''
    assistant = ''
    for msg in item.get('messages', []):
        role = (msg.get('role') or '').lower()
        content = msg.get('content') or ''
        if role == 'system' and not system:
            system = content
        elif role == 'user' and not user:
            user = content
        elif role == 'assistant' and not assistant:
            assistant = content
    records.append({
        'conversations': [
            {'from': 'human', 'value': user},
            {'from': 'gpt', 'value': assistant},
        ],
        'system': system,
    })

sharegpt_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

dataset_info_path = data_dir / 'dataset_info.json'
dataset_info = json.loads(dataset_info_path.read_text(encoding='utf-8'))
dataset_info['copypro_vi_sharegpt'] = {
    'file_name': 'copypro_vi_sharegpt.json',
    'formatting': 'sharegpt',
    'columns': {
        'messages': 'conversations',
        'system': 'system',
    },
    'tags': {
        'role_tag': 'from',
        'content_tag': 'value',
        'user_tag': 'human',
        'assistant_tag': 'gpt',
    },
}
dataset_info_path.write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding='utf-8')

print('Registered dataset copypro_vi_sharegpt with', len(records), 'samples')


In [ ]:
yaml_path = Path('/content/LLaMA-Factory/copypro_llama32_1b_lora_sft.yaml')
yaml_lines = [
    f'model_name_or_path: {BASE_MODEL}',
    'trust_remote_code: true',
    '',
    'stage: sft',
    'do_train: true',
    'finetuning_type: lora',
    'lora_rank: 16',
    'lora_alpha: 32',
    'lora_dropout: 0.05',
    'lora_target: all',
    '',
    'dataset: copypro_vi_sharegpt',
    'template: llama3',
    'cutoff_len: 2048',
    'max_samples: 100000',
    'overwrite_cache: true',
    'preprocessing_num_workers: 2',
    'dataloader_num_workers: 0',
    '',
    'output_dir: saves/copypro-llama32-1b/lora/sft',
    'logging_steps: 5',
    'save_steps: 50',
    'plot_loss: true',
    'overwrite_output_dir: true',
    'save_only_model: false',
    'report_to: none',
    '',
    'per_device_train_batch_size: 1',
    'gradient_accumulation_steps: 8',
    'learning_rate: 2.0e-4',
    'num_train_epochs: 2.0',
    'lr_scheduler_type: cosine',
    'warmup_ratio: 0.05',
    'fp16: true',
    'quantization_bit: 4',
]
yaml_path.write_text('
'.join(yaml_lines) + '
', encoding='utf-8')
print(yaml_path.read_text(encoding='utf-8'))


In [ ]:
%cd /content/LLaMA-Factory
!llamafactory-cli train copypro_llama32_1b_lora_sft.yaml


In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import os

adapter_dir = Path('/content/LLaMA-Factory/saves/copypro-llama32-1b/lora/sft')
assert adapter_dir.exists(), adapter_dir

api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=OUTPUT_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_folder(repo_id=OUTPUT_REPO, repo_type='model', folder_path=str(adapter_dir), token=os.environ['HF_TOKEN'])
print('Uploaded LoRA adapter to', f'https://huggingface.co/{OUTPUT_REPO}')


After training, your Hugging Face repo should contain adapter files such as `adapter_model.safetensors` and `adapter_config.json`.
